# Lesson 2: Testing and Metrics

In the previous notebook we saw that LLM outputs vary across runs. Now we tackle the harder question: **how do you measure whether the output is good?**

We'll work with a concrete task — extracting a plot summary and character names from a Sherlock Holmes story — and build up from observation to metrics to automated evaluation.

1. **Run the extraction** — see how outputs differ
2. **Think about metrics** — what makes an extraction "correct"?
3. **LLM-as-judge** — use a second LLM call to score the output
4. **Design your own judge** — exercise

Run cells in order.

In [ ]:
import os
import json
from pathlib import Path
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()
if not os.getenv("OPENAI_API_KEY"):
    raise RuntimeError("OPENAI_API_KEY not found. Add it to .env")

client = OpenAI()
MODEL = "gpt-4.1-mini"

text_path = Path("labs/02_standalone_agents/prompting/sherlock-short.txt")
if not text_path.exists():
    text_path = Path("prompting/sherlock-short.txt")
story = text_path.read_text()
print(f"Client ready. Story loaded: {len(story):,} characters")

## Step 1: Extract plot and characters using a Jinja2 prompt template

We load a **Jinja2 prompt template** from `prompting/plot_prompt_template.j2` that extracts the plot summary and character names as JSON. The template receives `story_text` as a variable.

In [ ]:
import re
from jinja2 import Template

# Load the Jinja2 prompt template from file
template_path = Path("labs/02_standalone_agents/solutions/prompting/plot_prompt_template.j2")
if not template_path.exists():
    template_path = Path("prompting/plot_prompt_template.j2")
prompt_template = Template(template_path.read_text())

# Render the template
rendered_prompt = prompt_template.render(story_text=story)

# Inspect the rendered prompt
print("=== Rendered prompt (first 500 chars) ===")
print(rendered_prompt[:500])
print("...\n")

def parse_json_response(text):
    """Extract JSON from an LLM response, handling markdown fences and surrounding text."""
    # Try direct parse first
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        pass
    # Strip markdown fences
    clean = text.strip().removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(clean)
    except json.JSONDecodeError:
        pass
    # Find first { ... } block
    match = re.search(r"\{.*\}", clean, re.DOTALL)
    if match:
        return json.loads(match.group())
    raise ValueError(f"Could not extract JSON from response:\n{text[:300]}")

# Run extraction
r = client.chat.completions.create(
    model=MODEL,
    messages=[{"role": "user", "content": rendered_prompt}],
    temperature=0.7,
)
raw = r.choices[0].message.content
result = parse_json_response(raw)

print("=== Model output ===")
print(f"Plot:       {result['plot_summary']}")
print(f"Characters: {result['characters']}")

### Inspect the output

Let's look at what the model extracted.

In [ ]:
# Quick look at what we got
print("Plot summary:")
print(f"  {result['plot_summary']}\n")
print("Characters found:")
for c in result["characters"]:
    print(f"  - {c}")

## Step 2: Think about metrics

Before we automate anything, we need to decide **what "good" means**. This is the hardest part of LLM evaluation — and it's a design decision, not a technical one.

### For character extraction (structured)

This is more tractable. If we have a ground-truth list, we can compute:
- **Precision** — of the names the model returned, how many are real characters?
- **Recall** — of the real characters, how many did the model find?
- **F1** — harmonic mean of precision and recall

### For plot summary (free text)

This is harder. A good summary should:
- Cover the **key events** (completeness)
- Not include events that **didn't happen** (faithfulness)
- Be **concise** (no filler)

There's no simple formula for this. Traditional metrics like BLEU or ROUGE compare word overlap with a reference, but they miss meaning — a perfect paraphrase scores poorly. This is where **LLM-as-judge** comes in.

### Character extraction: precision, recall, F1

Let's compute these against a ground-truth list.

In [ ]:
# Ground truth: characters actually mentioned in the passage
GROUND_TRUTH_CHARACTERS = {
    "Sherlock Holmes", "Watson", "Irene Adler",
}

def normalize(name):
    """Lowercase, strip titles and first-name-only variants."""
    return name.lower().strip()

def fuzzy_match(predicted, ground_truth):
    """Check if a predicted name matches any ground truth name (substring match)."""
    p = normalize(predicted)
    for gt in ground_truth:
        g = normalize(gt)
        if p in g or g in p:
            return True
    return False

predicted = result["characters"]
tp = sum(1 for p in predicted if fuzzy_match(p, GROUND_TRUTH_CHARACTERS))
precision = tp / len(predicted) if predicted else 0
recall = tp / len(GROUND_TRUTH_CHARACTERS)
f1 = 2 * precision * recall / (precision + recall) if (precision + recall) > 0 else 0
print(f"Predicted: {predicted}")
print(f"  Precision={precision:.2f}  Recall={recall:.2f}  F1={f1:.2f}")

## Step 3: LLM-as-judge for the plot summary

For free-text outputs, we can ask **another LLM call** to evaluate the quality. The idea:
1. Give the judge the original text and the model's summary
2. Ask it to score on specific criteria (completeness, faithfulness, conciseness)
3. Require a structured response (JSON with scores and reasoning)

This isn't perfect — the judge is also an LLM with its own biases — but it scales better than manual review and correlates well with human judgments in practice.

In [ ]:
JUDGE_PROMPT = """You are an expert literary judge. You will be given:
- An original text passage
- A plot summary written by a student

Score the summary on three criteria (1-5 each):

1. **Completeness**: Does the summary mention the key events and characters?
   - 5 = all major events covered
   - 3 = some events missing
   - 1 = most events missing

2. **Faithfulness**: Does the summary only include things that actually happen in the text?
   - 5 = everything is accurate
   - 3 = minor inaccuracies
   - 1 = major fabrications

3. **Conciseness**: Is the summary appropriately brief without unnecessary detail?
   - 5 = tight and focused
   - 3 = some filler
   - 1 = very verbose or repetitive

Return valid JSON only:
{"completeness": {"score": N, "reason": "..."}, "faithfulness": {"score": N, "reason": "..."}, "conciseness": {"score": N, "reason": "..."}}
"""

print("Judging the plot summary...\n")
judge_r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": JUDGE_PROMPT},
        {"role": "user", "content": f"ORIGINAL TEXT:\n{story[:2000]}\n\nSUMMARY:\n{result['plot_summary']}"},
    ],
    temperature=0,
)
scores = parse_json_response(judge_r.choices[0].message.content)

for criterion, data in scores.items():
    print(f"  {criterion:15s} {data['score']}/5 — {data['reason']}")

## Step 4: Exercise — design your own judge

The judge prompt above is just one way to evaluate summaries. **Your turn:**

1. Look at the 3 plot summaries above. Do you agree with the judge's scores? Where does it get it wrong?
2. Write your own judge prompt below. You might:
   - Change the criteria (e.g., add "readability" or "captures mystery/tension")
   - Change the scale (binary yes/no instead of 1-5)
   - Add few-shot examples of good vs bad summaries
   - Be more specific about what counts as a "key event" in this passage
3. Run your judge on the same 3 summaries and compare results.

**Bonus:** Can you make the judge *more consistent* across its own runs? (Hint: temperature, structured output, few-shot examples)

In [ ]:
# TODO: Write your own judge prompt
MY_JUDGE_PROMPT = """
YOUR JUDGE PROMPT HERE
"""

judge_r = client.chat.completions.create(
    model=MODEL,
    messages=[
        {"role": "system", "content": MY_JUDGE_PROMPT},
        {"role": "user", "content": f"ORIGINAL TEXT:\n{story[:2000]}\n\nSUMMARY:\n{result['plot_summary']}"},
    ],
    temperature=0,
)
print(judge_r.choices[0].message.content)